In [1]:
import pandas as pd
import torch
import time
import psutil
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import f1_score

# Load saved elements from Notebook 1
train_df = pd.read_csv('train_split.csv').dropna()
test_df = pd.read_csv('test_split.csv').dropna()

# FIX: Separate classes first to build a perfectly balanced training slice
toxic_train = train_df[train_df['label'] == 1].sample(n=5000, random_state=42, replace=True) # replace=True allows sampling if pool is small
nontoxic_train = train_df[train_df['label'] == 0].sample(n=5000, random_state=42)
train_balanced = pd.concat([toxic_train, nontoxic_train]).sample(frac=1, random_state=42).reset_index(drop=True)

# Separate classes for evaluation slice (1,000 toxic, 1,000 non-toxic)
toxic_test = test_df[test_df['label'] == 1].sample(n=1000, random_state=42, replace=True)
nontoxic_test = test_df[test_df['label'] == 0].sample(n=1000, random_state=42)
test_balanced = pd.concat([toxic_test, nontoxic_test]).sample(frac=1, random_state=42).reset_index(drop=True)

train_ds = Dataset.from_dict({"text": train_balanced['text'].tolist(), "label": train_balanced['label'].tolist()})
test_ds = Dataset.from_dict({"text": test_balanced['text'].tolist(), "label": test_balanced['label'].tolist()})

MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_test = test_ds.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 2000/2000 [00:00<00:00, 26345.63 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti
CUDA Version: 12.8


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(f"Using device: {device}")

Using device: cuda


In [7]:
import sys
print(sys.executable)

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\python.exe


In [4]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"f1": f1_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",        # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    fp16=torch.cuda.is_available(), # Uses GPU acceleration if available
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Training Transformer...")
trainer.train()
model.save_pretrained("distilbert_toxic_model")
tokenizer.save_pretrained("distilbert_toxic_model")

Training Transformer...


Epoch,Training Loss,Validation Loss,F1
1,No log,0.378964,0.836090


('distilbert_toxic_model\\tokenizer_config.json',
 'distilbert_toxic_model\\special_tokens_map.json',
 'distilbert_toxic_model\\vocab.txt',
 'distilbert_toxic_model\\added_tokens.json',
 'distilbert_toxic_model\\tokenizer.json')

In [6]:
# Benchmark Transformer Efficiency
process = psutil.Process(os.getpid())
start_mem = process.memory_info().rss / (1024 * 1024)

start_time = time.perf_counter()
predictions = trainer.predict(tokenized_test)
end_time = time.perf_counter()

end_mem = process.memory_info().rss / (1024 * 1024)
latency_per_sample = ((end_time - start_time) * 1000) / len(test_ds)

print(f"DistilBERT Final F1 Score: {predictions.metrics['test_f1']:.4f}")
print(f"DistilBERT Latency per Document: {latency_per_sample:.4f} ms")
print(f"DistilBERT Weight Storage Size: ~268 MB")

DistilBERT Final F1 Score: 0.8361
DistilBERT Latency per Document: 0.3415 ms
DistilBERT Weight Storage Size: ~268 MB
